# Sparsity Preserving QAT

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [1]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## Import Module

In [3]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

import tempfile

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [4]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


## Load Baseline Model for MNIST (CNN)

In [5]:
model = tf.keras.models.load_model('/content/drive/MyDrive/files/save/baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [6]:
_, baseline_model_accuracy = model.evaluate(test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


## Prune and fine-tune
* **purning schedule** : Constant Sparsity
    - sparsity : 0.5
    - begin_step : 0
    - frequency : every 100 step


In [7]:
pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.ConstantSparsity(0.5, begin_step=0, frequency=100)
  }

callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep()
]

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

pruned_model.compile(
  loss=keras.losses.SparseCategoricalCrossentropy(),
  optimizer=keras.optimizers.Adam(learning_rate=1e-5), # 작은 learning rate 적용
  metrics=['accuracy'])

pruned_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_reshap  (None, 28, 28, 1)         1         
 e (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_conv2d  (None, 26, 26, 32)        610       
  (PruneLowMagnitude)                                            
                                                                 
 prune_low_magnitude_max_po  (None, 13, 13, 32)        1         
 oling2d (PruneLowMagnitude                                      
 )                                                               
                                                                 
 prune_low_magnitude_conv2d  (None, 11, 11, 16)        9234      
 _1 (PruneLowMagnitude)                                          
                                                        

## Fine tune the model for pruning

In [8]:
pruned_model.fit(
  train_images,
  train_labels,
  epochs=3,
  validation_split=0.1,
  callbacks=callbacks)

Epoch 1/3
1688/1688 [==============================] - 42s 23ms/step - loss: 0.0167 - accuracy: 0.9952 - val_loss: 0.0405 - val_accuracy: 0.9898
Epoch 2/3
1688/1688 [==============================] - 38s 23ms/step - loss: 0.0115 - accuracy: 0.9971 - val_loss: 0.0377 - val_accuracy: 0.9907
Epoch 3/3
1688/1688 [==============================] - 40s 24ms/step - loss: 0.0093 - accuracy: 0.9978 - val_loss: 0.0364 - val_accuracy: 0.9908


## Sparsity 확인

In [9]:
def print_model_weights_sparsity(model):
    for layer in model.layers:
        if isinstance(layer, keras.layers.Wrapper):
            weights = layer.trainable_weights
        else:
            weights = layer.weights
        for weight in weights:
            if "quantize_layer" in weight.name:
                continue
            weight_size = weight.numpy().size
            zero_num = np.count_nonzero(weight == 0)
            print(
                f"{weight.name}: {zero_num/weight_size:.2%} sparsity ",
                f"({zero_num}/{weight_size})",
            )

In [10]:
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

print_model_weights_sparsity(stripped_pruned_model)

conv2d/kernel:0: 50.00% sparsity  (144/288)
conv2d/bias:0: 0.00% sparsity  (0/32)
conv2d_1/kernel:0: 50.00% sparsity  (2304/4608)
conv2d_1/bias:0: 0.00% sparsity  (0/16)
dense/kernel:0: 50.00% sparsity  (25600/51200)
dense/bias:0: 0.00% sparsity  (0/128)
dense_1/kernel:0: 50.00% sparsity  (640/1280)
dense_1/bias:0: 0.00% sparsity  (0/10)


## Accuracy Check : Baseline model VS Pruned model

In [11]:
_, pruned_model_accuracy = pruned_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Pruned test accuracy:', pruned_model_accuracy)

Baseline test accuracy: 0.9900000095367432
Pruned test accuracy: 0.9909999966621399


## QAT VS PQAT
* QAT : training 과정에서 sparsity 파괴
* PQAT : training 과정에서도 sparsity 유지

In [12]:
# QAT
qat_model = tfmot.quantization.keras.quantize_model(stripped_pruned_model)

qat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
print('Train QAT model:')
qat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Train QAT model:
422/422 [==============================] - 39s 90ms/step - loss: 0.0062 - accuracy: 0.9981 - val_loss: 0.0336 - val_accuracy: 0.9930


In [13]:
# PQAT
quant_aware_annotate_model = tfmot.quantization.keras.quantize_annotate_model(
              stripped_pruned_model)
pqat_model = tfmot.quantization.keras.quantize_apply(
              quant_aware_annotate_model,
              tfmot.experimental.combine.Default8BitPrunePreserveQuantizeScheme())

pqat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
print('Train PQAT Model:')
pqat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Train PQAT Model:
422/422 [==============================] - 36s 82ms/step - loss: 0.0055 - accuracy: 0.9987 - val_loss: 0.0326 - val_accuracy: 0.9927


## Sparsity 확인 : QAT model VS PQAT model

In [14]:
print("PQAT Model sparsity:")
print_model_weights_sparsity(pqat_model)
print()
print("QAT Model sparsity:")
print_model_weights_sparsity(qat_model)

PQAT Model sparsity:
conv2d/kernel:0: 50.00% sparsity  (144/288)
conv2d/bias:0: 0.00% sparsity  (0/32)
conv2d_1/kernel:0: 50.00% sparsity  (2304/4608)
conv2d_1/bias:0: 0.00% sparsity  (0/16)
dense/kernel:0: 50.00% sparsity  (25600/51200)
dense/bias:0: 0.00% sparsity  (0/128)
dense_1/kernel:0: 50.00% sparsity  (640/1280)
dense_1/bias:0: 0.00% sparsity  (0/10)

QAT Model sparsity:
conv2d/kernel:0: 8.68% sparsity  (25/288)
conv2d/bias:0: 0.00% sparsity  (0/32)
conv2d_1/kernel:0: 5.62% sparsity  (259/4608)
conv2d_1/bias:0: 0.00% sparsity  (0/16)
dense/kernel:0: 7.50% sparsity  (3842/51200)
dense/bias:0: 0.00% sparsity  (0/128)
dense_1/kernel:0: 7.34% sparsity  (94/1280)
dense_1/bias:0: 0.00% sparsity  (0/10)


## PQAT 모델의 benefit
Since this is a small model, the difference between the two models isn't very noticeable. Applying pruning and PQAT to a bigger production model would yield a more significant compression.

In [15]:
import zipfile
import os

def get_gzipped_model_size(file):
  # It returns the size of the gzipped model in kilobytes.

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(file)

  return os.path.getsize(zipped_file)/1000

In [16]:
litert_model_path = "/content/drive/MyDrive/files/save/"

In [17]:
import pathlib
models_dir = pathlib.Path(litert_model_path)
models_dir.mkdir(exist_ok=True)

# QAT model
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

qat_tflite_model = converter.convert()

qat_model_file = litert_model_path + 'mnist_pqat_x_model.tflite'
with open(qat_model_file, 'wb') as f:
    f.write(qat_tflite_model)

# PQAT model
converter = tf.lite.TFLiteConverter.from_keras_model(pqat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

pqat_tflite_model = converter.convert()

pqat_model_file = litert_model_path + 'mnist_pqat_model.tflite'
with open(pqat_model_file, 'wb') as f:
    f.write(pqat_tflite_model)

print("QAT model size: ", get_gzipped_model_size(qat_model_file), ' KB')
print("PQAT model size: ", get_gzipped_model_size(pqat_model_file), ' KB')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT model size:  46.978  KB
PQAT model size:  39.153  KB


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [18]:
def eval_model(interpreter):
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]
  input_index = input_details["index"]
  output_index = output_details["index"]

  # Run predictions on every image in the "test" dataset.
  prediction_digits = []
  for i, test_image in enumerate(test_images):
    # Check if the input type is quantized, then rescale input data to uint8
    if input_details['dtype'] == np.int8:
      input_scale, input_zero_point = input_details["quantization"]
      test_image = test_image / input_scale + input_zero_point

    test_image = np.expand_dims(test_image, axis=0).astype(input_details['dtype'])
    interpreter.set_tensor(input_index, test_image)

    # Run inference.
    interpreter.invoke()

    # Post-processing: remove batch dimension and find the digit with highest
    # probability.
    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  # Compare prediction results with ground truth labels to calculate accuracy.
  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [19]:
interpreter = tf.lite.Interpreter(pqat_model_file)
interpreter.allocate_tensors()

pqat_test_accuracy = eval_model(interpreter)

print('Pruned and quantized TFLite test_accuracy:', pqat_test_accuracy)
print('Baseline model test accuracy:', baseline_model_accuracy)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Pruned and quantized TFLite test_accuracy: 0.9923
Baseline model test accuracy: 0.9900000095367432
